In [5]:
import json
import pandas as pd
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification

In [6]:
# Load your test data from a JSON file
with open("Test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

In [7]:
# Convert to DataFrame if it's a list of dictionaries
if isinstance(test_data, list):
    test_df = pd.DataFrame(test_data)
else:
    # If it's already in a different format, adjust accordingly
    test_df = pd.DataFrame([test_data])
# Ensure the content field exists
if "content" not in test_df.columns and "text" in test_df.columns:
    test_df["content"] = test_df["text"]
elif "content" not in test_df.columns:
    # Try to identify the text field
    text_fields = [col for col in test_df.columns if any(x in col.lower() for x in ["text", "content", "resume"])]
    if text_fields:
        test_df["content"] = test_df[text_fields[0]]
    else:
        raise ValueError("Could not find text content column in the data")

In [8]:
# Load test texts
texts = test_df["content"].tolist()
print(f"Loaded {len(texts)} test samples")

Loaded 106 test samples


In [9]:
# Initialize tokenizer - use the same one you used for training
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [10]:
# Custom Dataset
class ResumeDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = self.texts[idx]
        # Handle empty or None text
        if not text or pd.isna(text):
            text = " "  # Use a space as fallback
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        # Remove the batch dimension that the tokenizer adds
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

In [11]:
# Create Dataset and DataLoader
test_dataset = ResumeDataset(texts, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=8)  # Smaller batch size for debugging

In [16]:
import torch
from transformers import BertConfig, BertForSequenceClassification
try:
    # Attempt to load directly with base config — likely to fail due to vocab mismatch
    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=43)
    model.load_state_dict(torch.load("fifth_epoch.pt", map_location="cpu"))
except Exception as e:
    print(f"❌ Error loading model directly: {e}")
    print("🔁 Trying alternative loading method...")
    # Correct loading using training config
    config = BertConfig(
        vocab_size=28996,       # ✅ must match training
        num_labels=43,          # ✅ number of classes
        hidden_size=768,
        num_attention_heads=12,
        num_hidden_layers=12
    )
    # Initialize model from config (not from pretrained weights)
    model = BertForSequenceClassification(config)
    # Load your saved model state
    checkpoint = torch.load("fifth_epoch.pt", map_location="cpu")
    # Handle state_dict or full model
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)
    print("✅ Model loaded successfully using custom config.")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


❌ Error loading model directly: Error(s) in loading state_dict for BertForSequenceClassification:
	size mismatch for bert.embeddings.word_embeddings.weight: copying a param with shape torch.Size([28996, 768]) from checkpoint, the shape in current model is torch.Size([30522, 768]).
🔁 Trying alternative loading method...
✅ Model loaded successfully using custom config.


In [4]:
# Send model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)
model.eval()
# Prediction loop
all_preds = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        try:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            # Don't use token_type_ids unless your model was trained with them
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
        except Exception as e:
            print(f"Error processing batch: {e}")
            # Print shapes for debugging
            print(f"Input IDs shape: {input_ids.shape}")
            print(f"Attention Mask shape: {attention_mask.shape}")
            continue
# Done 🎉
print("✅ Predictions complete!")
print(f"Total predictions: {len(all_preds)}")
print(f"First 10 predictions: {all_preds[:10]}")
# If you have label mappings, you can convert numeric predictions to labels
# label_map = {0: "Category1", 1: "Category2", ...}
# predicted_labels = [label_map[pred] for pred in all_preds]

Loaded 106 test samples


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Error loading model directly: Error(s) in loading state_dict for BertForSequenceClassification:
	size mismatch for bert.embeddings.word_embeddings.weight: copying a param with shape torch.Size([28996, 768]) from checkpoint, the shape in current model is torch.Size([30522, 768]).
Trying alternative loading method...
Using device: cuda


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
